In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.huggingface import HuggingFaceLLM

/home/no0ne/Documents/machine-learning/Generative-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!mkdir data

In [10]:
documents = SimpleDirectoryReader("./data/").load_data()

In [11]:
print(documents)

[Document(id_='f1f54321-1ccb-4a43-8213-971c08fad20b', embedding=None, metadata={'page_label': '1', 'file_name': 'MachineTranslationwithAttention.pdf', 'file_path': '/home/no0ne/Documents/machine-learning/Generative-ai/rag_application_with_llamaindex_mistral/data/MachineTranslationwithAttention.pdf', 'file_type': 'application/pdf', 'file_size': 482310, 'creation_date': '2025-09-12', 'last_modified_date': '2025-09-12'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='See discussions, stats, and author profiles for this publication at: https://www.researchgate.net/publication/355917108\nNeural Machine Translation with Attention\nTechnical Rep

In [12]:
# setup prompts - specific to StableLM
from llama_index.core import PromptTemplate

system_prompt = """[SYSTEM] # You are a Q&A assistant. Your goal is to answer questions as accurately as possible based on the instructions and context provided.
"""

# This will wrap the default prompts that are internal to llama-index
query_wrapper_prompt = PromptTemplate("<|USER|>{query_str}<|ASSISTANT|>")

In [ ]:
import torch

llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=256,
    system_prompt=system_prompt
    generate_kwargs={"temperature" : 0.7, "do_simple" : False}
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name="mistralai/Mistral-7b-Instuct-v0.1",
    model_name="mistralai/Mistral-7b-Instuct-v0.1",
    device_map="auto",
    stopping_ids=[50278,50279,50277,1,0],
    tokenizer_kwargs={"max_length": 4096}, 
)

In [ ]:
from llama_index.embeddings.huggingface import huggingFaceEmbegging
embed_model  = huggingFaceEmbegging(model_name = "sentence-transformers/all-mpnet-base-v2")

In [ ]:
from llama_index.core import VectorStoreIndex, ServiceContext
service_context = ServiceContext.from_defaults(
    chunk_size=1024,
    llm=llm,
    embed_model=embed_model
)


In [ ]:
index = VectorStoreIndex.from_documents(documents, service_context=service_context)

In [ ]:
query_engine = index.as_query_engine()

In [ ]:
query_engine.query("what is attention")